In [0]:
import yaml
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
%run ../../adb_local/Utilities/de_utilities

In [0]:
# yaml_path = "/Workspace/Users/shivec195@gmail.com/adb/config/schema_definition/bronze_schema/loan_defaulters.yml"
# file_path = "/Volumes/dltshiv/source/files/loan_defaulters/"

In [0]:
# read the Yaml file and store into variable
with open(yaml_path, "r") as file:
    yaml_file = yaml.safe_load(file)

#retrieving the variables from YAML file
schema_name =yaml_file.get("schema_name")
table_name  =yaml_file.get("table_name")
file_name   =yaml_file.get("file_name")
column_list =yaml_file.get("columns",[])

# create a custom schema from YAML file
modify_schema = StructType(
    [
        StructField(column["source_column"],StringType(),True) for column in column_list
    ]
)

#Reading the CSV file and loading into the dataframe 
df_default_loan = spark.read.format("csv").option("header","true").schema(modify_schema).load(path+file_name)

In [0]:
# creating column mapping for destination columns
column_mapping = [(column["source_column"],column["destination_column"]) for column in column_list]

In [0]:
#renameing dataframe columns as per the destination columns
df_default_loan_table = df_default_loan.select([col(column[0]).alias(column[1]) for column in column_mapping])

#preparing final table name
final_table_name = "bronze."+schema_name+"."+table_name

#writing the dataframe to the table
df_default_loan_table.write.mode("append").insertInto(final_table_name)
# df_default_loan_table.write.mode("overwrite").saveAsTable(final_table_name)

In [0]:
%sql
select * from bronze.bronze_schema.loan_defaulters limit 2

In [0]:
print("total_records :", df_default_loan_table.count())
print("total_distinct_records :", df_default_loan_table.distinct().count())
print("distinct_records_basied_on_column_id :",df_default_loan_table.select("id").distinct().count())
print("distinct_count_without_id_column :",df_default_loan_table.select([each_column for each_column in df_default_loan_table.columns if each_column !="id"]).distinct().count())
df_default_loan_table.groupBy("id").agg(count("id").alias("cnt")).filter(col("cnt")>1).show()
print("How many are values are duplicate :", df_default_loan_table.groupBy("id").agg(count("id").alias("cnt")).filter(col("cnt")>1).count())

In [0]:
df_null=df_default_loan_table.select([count(when(col(each_column).isNull(),each_column)).alias(each_column) for each_column in df_default_loan_table.columns])
df_null.select([each_column for each_column in df_null.columns if df_null.select(each_column).first()[each_column]>0]).display()

In [0]:
df_default_loan_table.fillna({"employment_type":"No Data"}).select("employment_type").filter(col("employment_type")=="No Data").count()
#Homework --> convert above statement in dynamic way

In [0]:
numeric_columns=["disbursed_value","asset_cost","Loan to value","total_balance_outstanding","total_sanctioned_amount","total_disbursed_amount"]
boundaries={}
for each_numeric_column in numeric_columns:
    quantiles=df_default_loan.withColumn(each_numeric_column,col(each_numeric_column).cast('float')).approxQuantile(each_numeric_column,[0.25,0.75],0.05)
    print(each_numeric_column,quantiles)
    iqr = quantiles[1]-quantiles[0]
    print(each_numeric_column,iqr)
    boundaries[each_numeric_column]=[quantiles[1]-(1.5*iqr),quantiles[0]+(1.5*iqr)]
    print(each_numeric_column,boundaries[each_numeric_column])

In [0]:
boundaries

In [0]:
df_default_loan.select(
    *[col(each_nc) for each_nc in numeric_columns],
    *[when(col(nc).between(boundaries[nc][0],boundaries[nc][1]),0).otherwise(1).alias(nc+"_outlier") for nc in numeric_columns]
).display()